# Geospatial Visualization with Python

This notebook shows three essential visualization patterns: **raster data**, **vector data**, and **combining raster and vector** layers. We use `rasterio` and `matplotlib` for rasters, and `geopandas` for vectors.

In [ ]:
# Optional: Install packages if needed
# %pip install geopandas rasterio matplotlib

import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show as rasterio_show
import geopandas as gpd
from pathlib import Path
import numpy as np

## Example 1: Visualizing Raster Data

Raster visualization displays pixel values as colors. For single-band data (e.g., elevation), we use a colormap. For multi-band RGB imagery, we display bands 1–3 as red, green, and blue.

In [ ]:
# Create a sample RGB raster (or use a real GeoTIFF)
data_dir = Path("../data")
data_dir.mkdir(exist_ok=True)
raster_path = data_dir / "sample_raster.tif"

# Use a projected CRS (UTM 10N) so bounds are in meters
crs = "EPSG:32610"
xmin, ymin, xmax, ymax = 500000, 4100000, 500640, 4100640  # 640m x 640m box

# Synthetic 3-band image (64x64 pixels)
np.random.seed(42)
r = np.clip(np.random.rand(64, 64) * 200 + 55, 0, 255).astype(np.uint8)
g = np.clip(np.random.rand(64, 64) * 180 + 75, 0, 255).astype(np.uint8)
b = np.clip(np.random.rand(64, 64) * 220 + 35, 0, 255).astype(np.uint8)
rgb = np.stack([r, g, b], axis=0)

transform = rasterio.transform.from_bounds(xmin, ymin, xmax, ymax, 64, 64)
with rasterio.open(
    raster_path, "w", driver="GTiff", height=64, width=64,
    count=3, dtype=np.uint8, crs=crs, transform=transform
) as dst:
    dst.write(rgb[0], 1)
    dst.write(rgb[1], 2)
    dst.write(rgb[2], 3)

In [ ]:
# Visualize the raster
fig, ax = plt.subplots(figsize=(8, 8))
with rasterio.open(raster_path) as src:
    rasterio_show(src, ax=ax)
ax.set_title("Example 1: Raster visualization (RGB)")
plt.tight_layout()
plt.show()

**Interpretation**: The raster is displayed with geographic axes. `rasterio.plot.show` automatically uses the raster's transform to place pixels in the correct coordinate space. For real satellite imagery, you would see the actual landscape.

## Example 2: Visualizing Vector Data

Vector visualization draws points, lines, and polygons on a map. We can color features by an attribute (e.g., population, land use) or use a single style for all features.

In [ ]:
from shapely.geometry import Point, Polygon

# Create sample vector data in same CRS and extent as raster (UTM meters)
points = [
    Point(500100, 4100100), Point(500200, 4100300), Point(500400, 4100150),
    Point(500300, 4100400), Point(500500, 4100350)
]
polygons = [
    Polygon([(500050, 4100050), (500150, 4100050), (500150, 4100150), (500050, 4100150)]),
    Polygon([(500350, 4100250), (500450, 4100250), (500450, 4100400), (500350, 4100400)]),
]
gdf_points = gpd.GeoDataFrame(
    {"id": range(len(points)), "value": [10, 20, 15, 25, 30]},
    geometry=points, crs=crs
)
gdf_polygons = gpd.GeoDataFrame(
    {"id": range(len(polygons)), "type": ["A", "B"]},
    geometry=polygons, crs=crs
)

In [ ]:
# Visualize vector data
fig, ax = plt.subplots(figsize=(8, 8))
gdf_polygons.plot(ax=ax, facecolor="lightblue", edgecolor="darkblue", alpha=0.6)
gdf_points.plot(ax=ax, column="value", cmap="YlOrRd", legend=True, markersize=100)
ax.set_title("Example 2: Vector visualization (polygons + points colored by attribute)")
ax.set_xlabel("X")
ax.set_ylabel("Y")
plt.tight_layout()
plt.show()

**Interpretation**: Polygons are drawn with fill and edge colors. Points are colored by the `value` attribute using a colormap. This pattern is useful for choropleth maps and thematic overlays.

## Example 3: Combining Raster and Vector Data

Overlaying vector features on a raster base map is a common workflow: e.g., building footprints on satellite imagery, or administrative boundaries on a land cover map. Both layers must share the same CRS.

In [ ]:
# Vectors already use same CRS as raster; no reprojection needed
gdf_polygons_match = gdf_polygons
gdf_points_match = gdf_points

In [ ]:
# Combine raster and vector on the same axes
fig, ax = plt.subplots(figsize=(8, 8))
with rasterio.open(raster_path) as src:
    rasterio_show(src, ax=ax)
gdf_polygons_match.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=2)
gdf_points_match.plot(ax=ax, color="yellow", markersize=80, edgecolor="black")
ax.set_title("Example 3: Raster + vector overlay")
plt.tight_layout()
plt.show()

**Interpretation**: The raster provides the base imagery, and vector layers are drawn on top. Red polygon outlines and yellow points show how vector features align with the raster. In real applications, you might overlay building footprints on aerial imagery or roads on a land cover map.

## Summary

| Example | Tools | Use case |
|---------|-------|----------|
| Raster | `rasterio.plot.show`, `matplotlib` | Satellite imagery, elevation, land cover |
| Vector | `geopandas.plot`, `matplotlib` | Points, lines, polygons with attributes |
| Combined | Raster base + vector overlay | Contextual maps, validation, analysis |